# PCA Analysis: Dopamine & GABA Dynamics during Stimulus-Reward Learning

Config-driven PCA dimensionality reduction across datasets (SpontFB, CRFB, ToneFB) and neuron populations (Dopamine, GABA, Combined).

**Outputs**: PNG figures, manifest JSONs, and a summary CSV under `outputs/`.

In [ ]:
import os
import pandas as pd
from plot_pca import plot_pca, run_analysis

# Check kaleido for PNG export
try:
    import kaleido
    print(f"kaleido {kaleido.__version__} available for PNG export")
except ImportError:
    print("WARNING: kaleido not installed. PNG export will fail. Run: pip install kaleido")

## Configuration

Edit the config dict below to control which datasets, neuron combinations, and parameters are used.

In [ ]:
config = {
    # ---- Dataset definitions ----
    "datasets": {
        "SpontFB": {"mat_file": "dataSpontFB.mat", "var_name": "dataSpontFB"},
        "CRFB":    {"mat_file": "dataCRFB.mat",    "var_name": "dataCRFB"},
        "ToneFB":  {"mat_file": "dataToneFB.mat",  "var_name": "dataToneFB"},
    },

    # ---- Neuron combination definitions ----
    "neuron_combos": {
        "Dopamine": ["DF", "DB"],
        "GABA":     ["GF", "GB"],
        "Combined": ["DF", "DB", "GF", "GB"],
    },

    # ---- PCA parameters ----
    "pca": {"n_components": 3},

    # ---- Windowing parameters ----
    "window": {
        "event_idx": 600,   # index of the event within each half
        "window": 100,      # +/- timesteps around event (>=100 for ToneFB Reward)
        "dt": 0.01,         # seconds per timestep
    },

    # ---- Smoothing parameters ----
    "smoothing": {"sg_window": 11, "sg_order": 3},

    # ---- Visualization ----
    "visualization": {
        "fwd_cmap": "YlOrRd",
        "bwd_cmap": "Blues",
        "fig_width": 1100,
        "fig_height": 800,
        "plot_types": ["scatter", "trajectory"],
    },

    # ---- Output ----
    "output": {
        "base_dir": "outputs",
        "save_png": True,
        "save_manifest": True,
        "save_summary_csv": True,
        "show_figures": False,   # set True to display interactive plots
    },
}

## Run Full Analysis Pipeline

Iterates all dataset x neuron-combo combinations, generates figures, saves PNGs + manifests, and produces a summary CSV.

In [ ]:
summary = run_analysis(config)
print(f"Completed. Summary CSV: {summary['summary_csv_path']}")

## Summary Table

Displays all generated files and explained-variance metrics.

In [ ]:
summary_df = pd.DataFrame(summary['results'])
display(summary_df[[
    'dataset', 'neuron_combo', 'neuron_groups', 'n_neurons',
    'explained_var_pc1', 'explained_var_pc2', 'explained_var_pc3',
    'explained_var_total', 'status', 'warnings', 'error'
]])

## Inspect Individual Results

Display a specific figure interactively by selecting dataset and neuron combo.

In [ ]:
# Choose which result to inspect
dataset_key = "SpontFB"
combo_key = "Dopamine"

key = f"{dataset_key}_{combo_key}"
if key in summary['figures']:
    print(f"Showing: {key} - scatter")
    summary['figures'][key]['scatter'].show()
    print(f"Showing: {key} - trajectory")
    summary['figures'][key]['trajectory'].show()
else:
    print(f"Key '{key}' not found. Available: {list(summary['figures'].keys())}")

## Single Dataset Exploration

Use `plot_pca()` directly for a single dataset/combo with interactive display.

In [ ]:
result = plot_pca(
    mat_file="dataSpontFB.mat",
    var_name="dataSpontFB",
    dataset_name="SpontFB",
    neuron_groups=["DF", "DB"],
    neuron_combo_label="Dopamine",
    output_dir=None,   # no PNG saving for exploration
    show=True,
)
print(f"Explained variance: {result['explained_variance_ratio']}")
print(f"Total: {sum(result['explained_variance_ratio']):.4f}")
print(f"Neurons used: {result['n_neurons']} ({result['neuron_groups_used']})")
if result['warnings']:
    print(f"Warnings: {result['warnings']}")